# Day 3 - Conversational AI - aka Chatbot!

## The Use of Prompts

1. System Prompt - establish ground-rules, like "if you don't know the answer, just say so" provide critical background context

2. Context - during the conversation, insert context to give more relevant background information pertaining to the topic

3. Multi-Shot Prompting - provide example conversations to prime for specific scenarios, train on conversational style amnd demonstrate complex interactions

In [1]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"OpenAI API Key exists and begins {groq_api_key[:8]}")
else:
    print("OpenAI API Key not set")

OpenAI API Key exists and begins gsk_OyAI


In [3]:
# Initialize
groq_url = "https://api.groq.com/openai/v1"
MODEL="llama-3.3-70b-versatile"

groq = OpenAI(api_key = groq_api_key, base_url = groq_url)

In [4]:
# Again, I'll be in scientist-mode and change this global during the lab

system_message = "You are a helpful assistant"

## And now, writing a new callback

We now need to write a function called:

`chat(message, history)`

Which will be a callback function we will give gradio.

### The job of this function

Take a message, take the prior conversation, and return the response.


In [5]:
def chat(message, history):
    return "bananas"

In [6]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
def chat(message, history):
    return f"You said {message} and the history is {history} but I still say bananas"

In [8]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


## OK! Let's write a slightly better chat callback!

In [9]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content


In [10]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [11]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    stream = groq.chat.completions.create(model=MODEL, messages=messages, stream=True)
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [12]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


## OK let's keep going!

Using a system message to add context, and to give an example answer.. this is "one shot prompting" again

In [13]:
system_message = "You are a helpful assistant in a clothes store. You should try to gently encourage \
the customer to try items that are on sale. Hats are 60% off, and most other items are 50% off. \
For example, if the customer says 'I'm looking to buy a hat', \
you could reply something like, 'Wonderful - we have lots of hats - including several that are part of our sales event.'\
Encourage the customer to buy hats if they are unsure what to get."

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


In [15]:
system_message += "\nIf the customer asks for shoes, you should respond that shoes are not on sale today, \
but remind the customer to look at hats!"

In [16]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


In [17]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = groq.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [18]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


In [19]:
system_message += "\nIf the customer asks for kids section, you should respond that kids section is not available till now but we will add very soon, \
but remind the customer to look at hats!"

In [20]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


<table style="margin: 0; text-align: left;">
    <tr>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Conversational Assistants are of course a hugely common use case for Gen AI, and the latest frontier models are remarkably good at nuanced conversation. And Gradio makes it easy to have a user interface. Another crucial skill we covered is how to use prompting to provide context, information and examples.
<br/><br/>
Consider how you could apply an AI Assistant to your business, and make yourself a prototype. Use the system prompt to give context on your business, and set the tone for the LLM.</span>
        </td>
    </tr>
</table>

## AI Shopping Assistant

In [21]:
# A manual dataset..

products = [
    {"id": 1, "name": "Nike Air Zoom Pegasus", "price": 6500, "category": "shoes", "brand": "Nike", "rating": 4.5, "discount": 20, "stock": 12},
    {"id": 2, "name": "Adidas Ultraboost 22", "price": 9000, "category": "shoes", "brand": "Adidas", "rating": 4.7, "discount": 15, "stock": 8},
    {"id": 3, "name": "Puma Running Pro", "price": 4000, "category": "shoes", "brand": "Puma", "rating": 4.2, "discount": 25, "stock": 20},
    {"id": 4, "name": "Bata Formal Leather Shoes", "price": 2500, "category": "shoes", "brand": "Bata", "rating": 4.0, "discount": 10, "stock": 15},

    {"id": 5, "name": "Nike Sports Cap", "price": 1200, "category": "hats", "brand": "Nike", "rating": 4.3, "discount": 40, "stock": 25},
    {"id": 6, "name": "Adidas Baseball Cap", "price": 1000, "category": "hats", "brand": "Adidas", "rating": 4.4, "discount": 35, "stock": 18},
    {"id": 7, "name": "Puma Snapback Hat", "price": 900, "category": "hats", "brand": "Puma", "rating": 4.1, "discount": 50, "stock": 10},

    {"id": 8, "name": "Levi's Slim Fit Jeans", "price": 3000, "category": "clothing", "brand": "Levi's", "rating": 4.6, "discount": 20, "stock": 14},
    {"id": 9, "name": "H&M Casual T-Shirt", "price": 800, "category": "clothing", "brand": "H&M", "rating": 4.2, "discount": 30, "stock": 30},
    {"id": 10, "name": "Zara Formal Shirt", "price": 2200, "category": "clothing", "brand": "Zara", "rating": 4.5, "discount": 15, "stock": 9},

    {"id": 11, "name": "Fossil Analog Watch", "price": 7000, "category": "accessories", "brand": "Fossil", "rating": 4.7, "discount": 25, "stock": 6},
    {"id": 12, "name": "Casio G-Shock", "price": 5000, "category": "accessories", "brand": "Casio", "rating": 4.8, "discount": 10, "stock": 11},
    {"id": 13, "name": "Ray-Ban Sunglasses", "price": 6000, "category": "accessories", "brand": "Ray-Ban", "rating": 4.6, "discount": 20, "stock": 7},

    {"id": 14, "name": "Wildcraft Backpack", "price": 1800, "category": "bags", "brand": "Wildcraft", "rating": 4.3, "discount": 35, "stock": 22},
    {"id": 15, "name": "Skybags Travel Bag", "price": 2500, "category": "bags", "brand": "Skybags", "rating": 4.4, "discount": 30, "stock": 16},

    {"id": 16, "name": "Woodland Leather Belt", "price": 1500, "category": "belt", "brand": "Woodland", "rating": 4.2, "discount": 5, "stock": 13},
    {"id": 17, "name": "Titan Formal Belt", "price": 1200, "category": "belt", "brand": "Titan", "rating": 4.1, "discount": 10, "stock": 19},
]

In [22]:
# Smart Retrieval

def find_products(user_message):
    msg = user_message.lower()
    results = []

    for p in products:
        if p["category"] in msg or p["brand"].lower() in msg:
            results.append(p)

# Sort by Rating + discount

    results = sorted(results, key=lambda x: (-x["rating"], -x["discount"]))

    return results[:5]

In [23]:
# User memory

user_memory = {}

def update_memory(message):
    msg = message.lower()

    if "cheap" in msg or "budget" in msg:
        user_memory["price"] = "low"
    elif "premium" in msg or "expensive" in msg:
        user_memory["price"] = "high"

In [24]:
# Intent Detection

def detect_intent(message):
    msg = message.lower()

    if "buy" in msg or "recommend" in msg:
        return "shopping"
    elif "price" in msg:
        return "price_query"
    else:
        return "general"

In [25]:
# Create a chat function (core logic)

def chat(message, history):
    update_memory(message)

    relevant_products = find_products(message)

    # Apply memory filter
    if user_memory.get("price") == "low":
        relevant_products = sorted(relevant_products, key=lambda x: x["price"])
    elif user_memory.get("price") == "high":
        relevant_products = sorted(relevant_products, key=lambda x: -x["price"])

    # Format product info
    product_info = ""
    for p in relevant_products:
        product_info += (
            f"{p['name']} ({p['brand']})\n"
            f"Price: ₹{p['price']} | Discount: {p['discount']}% | Rating: ⭐{p['rating']}\n"
            f"Stock left: {p['stock']}\n\n"
        )

    system_message = f"""
    You are a smart AI shopping assistant.

    Available products:
    {product_info}

    Rules:
    - Recommend products based on user needs
    - Highlight discounts and ratings
    - Create urgency using stock info
    - Be friendly and persuasive
    """

    messages = [{"role": "system", "content": system_message}]

    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    messages.append({"role": "user", "content": message})

    stream = groq.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    response = ""
    for chunk in stream:
        content = chunk.choices[0].delta.content or ""
        response += content
        yield response

In [26]:
# Create a groq Interface.

gr.ChatInterface(
    fn=chat,
    title="🛍️ AI Shopping Assistant",
    description="Ask me to recommend products, compare items, or find deals!",
    theme="soft"
).launch()

c:\Users\KIIT\projects\llm_engineering\.venv\Lib\site-packages\gradio\chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


## Data Analyst Assistant

In [27]:
#Imports

import pandas as pd
from groq import Groq
import os
from dotenv import load_dotenv

In [28]:
#load API key

load_dotenv(override=True)
groq_api_key = os.getenv('GROQ_API_KEY')

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")

Groq API Key exists and begins gsk_OyAI


In [29]:
#Connect to Groq

groq_url = "https://api.groq.com/openai/v1"

groq = OpenAI(api_key=groq_api_key, base_url=groq_url)

In [30]:
!pip install openpyxl


[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
#Load dataset

file_path = "Subscription Pricing, Consumer Satisfaction, and Retention in OTT Platforms_ A Comparative Study of Netflix, Amazon Prime Video, and Disney+ Hotstar  (Responses).xlsx"

df = pd.read_excel(file_path)

df.head()

,Timestamp,Age Group,Gender,Occupation,Monthly Personal Income (INR),Do you share your OTT subscription with others?,Which OTT platforms do you currently subscribe to?,Which is your primary OTT platform?,Duration of OTT subscription,Average monthly spending on OTT subscriptions (INR),...,Social media trends influence what I watch on OTT.,OTT platforms provide better content variety compared to traditional television.,My television watching time has reduced after I started using OTT platforms.,I prefer OTT platforms over going to cinema halls for entertainment.,I rely primarily on OTT platforms for my entertainment needs.,Have you ever cancelled an OTT subscription?,"If yes, what was the main reason?",I intend to continue my current OTT subscription for the next year.,I am willing to switch to another platform offering lower price for similar content.,I would recommend my current OTT platform to others.
0,2026-02-21 13:46:16.176,18-24,Female,Salaried Employee,"25,001-50,000","Yes, with family","Netflix, Amazon Prime Video, Disney+ Hotstar",Netflix,1-3 years,200-500,...,4,5,5,2,4,No,NaN,4,4,4
1,2026-02-21 13:50:26.771,45 and above,Female,Self Employed,"25,001-50,000","Yes, with family",Zee5,Zee5,Less than 6 months,200-500,...,5,2,4,2,5,Yes,High price,2,5,5
2,2026-02-21 13:55:40.340,18-24,Male,Salaried Employee,"Above 1,00,000",No,"Netflix, Amazon Prime Video, Disney+ Hotstar, ...",Netflix,More than 3 years,Above 1200,...,5,5,5,5,5,No,NaN,5,1,5
3,2026-02-21 13:55:43.652,Below 18,Male,Salaried Employee,"Above 1,00,000",No,"Netflix, Amazon Prime Video, Disney+ Hotstar",Disney+ Hotstar,More than 3 years,Above 1200,...,4,3,5,1,1,Yes,"Poor content, Better competitor available",1,3,5
4,2026-02-21 13:59:10.006,18-24,Female,Salaried Employee,"25,001-50,000",No,NaN,Disney+ Hotstar,Less than 6 months,Below 200,...,2,2,4,2,3,Yes,"High price, Poor content",1,4,2


In [32]:
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()

In [33]:
for col in df.columns:
    print(col)

Timestamp
Age Group
Gender
Occupation
Monthly Personal Income (INR)
Do you share your OTT subscription with others?
Which OTT platforms do you currently subscribe to?
Which is your primary OTT platform?
Duration of  OTT subscription
Average monthly spending on OTT subscriptions (INR)
Average hours spent on OTT per week (Type the number)
My current OTT subscription is affordable relative to the benefits I receive.
I would cancel my subscription if the subscription price increases significantly.
I actively compare subscription prices across OTT platforms before renewing.
I prefer annual subscription plans due to lower effective pricing.
Discounts and promotional offers influence my subscription decision.
Exclusive or original content motivates me to remain subscribed.
Streaming quality and user interface significantly affect my subscription.
Availability of regional language content influences my subscription decision.
Social media trends influence what I watch on OTT.
OTT platforms prov

In [34]:
afford_col = [col for col in df.columns if "afford" in col.lower()][0]
content_col = [col for col in df.columns if "exclusive" in col.lower() or "content" in col.lower()][0]
ux_col = [col for col in df.columns if "quality" in col.lower() or "interface" in col.lower()][0]
retention_col = [col for col in df.columns if "continue" in col.lower()][0]
recommend_col = [col for col in df.columns if "recommend" in col.lower()][0]

print(afford_col)
print(content_col)
print(ux_col)
print(retention_col)
print(recommend_col)

My current OTT subscription is affordable relative to the benefits I receive.
Exclusive or original content motivates me to remain subscribed.
Streaming quality and user interface significantly affect my subscription.
I intend to continue my current OTT subscription for the next year.
I would recommend my current OTT platform to others.


In [35]:
#Rename columns

df = df.rename(columns={
    afford_col: "Affordability",
    content_col: "Content_Driven",
    ux_col: "UX_Impact",
    retention_col: "Retention",
    recommend_col: "Recommendation"
})

In [36]:
# Columns that contain Likert-scale responses
likert_cols = [
    'Affordability',
    'Content_Driven',
    'UX_Impact',
    'Retention',
    'Recommendation'
]

# Step 1: Clean text (important in real datasets)
for col in likert_cols:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Step 2: Mapping function
def map_likert(x):
    if 'strongly disagree' in x:
        return 1
    elif 'disagree' in x:
        return 2
    elif 'neutral' in x:
        return 3
    elif 'agree' in x and 'strongly' not in x:
        return 4
    elif 'strongly agree' in x:
        return 5
    else:
        return None

# Step 3: Apply mapping
for col in likert_cols:
    df[col] = df[col].apply(map_likert)

In [38]:
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()

In [39]:
for col in df.columns:
    print(col)

Timestamp
Age Group
Gender
Occupation
Monthly Personal Income (INR)
Do you share your OTT subscription with others?
Which OTT platforms do you currently subscribe to?
Which is your primary OTT platform?
Duration of  OTT subscription
Average monthly spending on OTT subscriptions (INR)
Average hours spent on OTT per week (Type the number)
My current OTT subscription is affordable relative to the benefits I receive.
I would cancel my subscription if the subscription price increases significantly.
I actively compare subscription prices across OTT platforms before renewing.
I prefer annual subscription plans due to lower effective pricing.
Discounts and promotional offers influence my subscription decision.
Exclusive or original content motivates me to remain subscribed.
Streaming quality and user interface significantly affect my subscription.
Availability of regional language content influences my subscription decision.
Social media trends influence what I watch on OTT.
OTT platforms prov

In [41]:
df.columns = df.columns.str.strip()

col_map = {}

for col in df.columns:
    c = col.lower()

    if "afford" in c:
        col_map[col] = "Affordability"

    elif "exclusive" in c or "content motivates" in c:
        col_map[col] = "Content_Driven"

    elif "quality" in c or "interface" in c:
        col_map[col] = "UX_Impact"

    elif "continue" in c:
        col_map[col] = "Retention"

    elif "recommend" in c:
        col_map[col] = "Recommendation"

# apply rename
df = df.rename(columns=col_map)

In [42]:
print(df.columns)

Index(['Timestamp', 'Age Group', 'Gender', 'Occupation',
       'Monthly Personal Income (INR)',
       'Do you share your OTT subscription with others?',
       'Which OTT platforms do you currently subscribe to?',
       'Which is your primary OTT platform?', 'Duration of  OTT subscription',
       'Average monthly spending on OTT subscriptions (INR)',
       'Average hours spent on OTT per week (Type the number)',
       'Affordability',
       'I would cancel my subscription if the subscription price increases significantly.',
       'I actively compare subscription prices across OTT platforms before renewing.',
       'I prefer annual subscription plans due to lower effective pricing.',
       'Discounts and promotional offers influence my subscription decision.',
       'Content_Driven', 'UX_Impact',
       'Availability of regional language content influences my subscription decision.',
       'Social media trends influence what I watch on OTT.',
       'OTT platforms provide be

In [43]:
print(df['Affordability'].unique())
print(df['Content_Driven'].unique())

[3 2 5 4 1]
[4 5 1 3 2]


In [44]:
df[['Affordability','Content_Driven','UX_Impact','Retention','Recommendation']].describe()

,Affordability,Content_Driven,UX_Impact,Retention,Recommendation
count,100.000000,100.00000,100.000000,100.000000,100.000000
mean,3.560000,3.80000,3.630000,3.550000,3.760000
std,1.148737,1.13707,1.142874,1.225775,1.120245
min,1.000000,1.00000,1.000000,1.000000,1.000000
25%,3.000000,3.00000,3.000000,3.000000,3.000000
50%,4.000000,4.00000,4.000000,4.000000,4.000000
75%,4.000000,5.00000,5.000000,4.000000,5.000000
max,5.000000,5.00000,5.000000,5.000000,5.000000


In [45]:
df = df.rename(columns={
    'Which is your primary OTT platform?': 'Platform'
})

print(df['Platform'].head())

0            Netflix
1               Zee5
2            Netflix
3    Disney+ Hotstar
4    Disney+ Hotstar
Name: Platform, dtype: object


In [46]:
# Clean all column names first
df.columns = df.columns.str.strip()

# Create mapping dynamically
col_map = {}

for col in df.columns:
    c = col.lower()

    if "primary" in c and "platform" in c:
        col_map[col] = "Platform"

    elif "spending" in c:
        col_map[col] = "Spending"

    elif "hours" in c:
        col_map[col] = "Hours"

    elif "afford" in c:
        col_map[col] = "Affordability"

    elif "exclusive" in c or "content motivates" in c:
        col_map[col] = "Content_Driven"

    elif "quality" in c or "interface" in c:
        col_map[col] = "UX_Impact"

    elif "continue" in c:
        col_map[col] = "Retention"

    elif "recommend" in c:
        col_map[col] = "Recommendation"

    elif "cancelled" in c:
        col_map[col] = "Cancelled"

# Apply rename
df = df.rename(columns=col_map)

print("Final columns:")
print(df.columns)

Final columns:
Index(['Timestamp', 'Age Group', 'Gender', 'Occupation',
       'Monthly Personal Income (INR)',
       'Do you share your OTT subscription with others?',
       'Which OTT platforms do you currently subscribe to?', 'Platform',
       'Duration of  OTT subscription', 'Spending', 'Hours', 'Affordability',
       'I would cancel my subscription if the subscription price increases significantly.',
       'I actively compare subscription prices across OTT platforms before renewing.',
       'I prefer annual subscription plans due to lower effective pricing.',
       'Discounts and promotional offers influence my subscription decision.',
       'Content_Driven', 'UX_Impact',
       'Availability of regional language content influences my subscription decision.',
       'Social media trends influence what I watch on OTT.',
       'OTT platforms provide better content variety compared to traditional television.',
       'My television watching time has reduced after I started u

In [47]:
required_cols = [
    'Platform','Spending','Hours',
    'Affordability','Content_Driven',
    'UX_Impact','Retention','Recommendation'
]

for col in required_cols:
    print(col, "→", col in df.columns)

Platform → True
Spending → True
Hours → True
Affordability → True
Content_Driven → True
UX_Impact → True
Retention → True
Recommendation → True


In [48]:
df['Spending'] = pd.to_numeric(df['Spending'], errors='coerce')
df['Hours'] = pd.to_numeric(df['Hours'], errors='coerce')

In [49]:
#Create data context

def advanced_context(df):
    context = ""

    context += f"Total responses: {len(df)}\n\n"

    context += "Platform usage:\n"
    context += df['Platform'].value_counts().to_string() + "\n\n"

    context += f"Average spending: ₹{df['Spending'].mean():.2f}\n\n"
    context += f"Average hours watched: {df['Hours'].mean():.2f}\n\n"

    context += "Scores by platform:\n"
    context += df.groupby('Platform')[
        ['Affordability','Content_Driven','UX_Impact','Retention','Recommendation']
    ].mean().to_string() + "\n\n"

    return context

data_context = advanced_context(df)
print(data_context)

Total responses: 100

Platform usage:
Platform
Netflix               38
Disney+ Hotstar       32
Amazon Prime Video    20
Zee5                   3
Hoichoi, Hotstar       1
YouTube                1
None, only Youtube     1
No ott platform        1

Average spending: ₹nan

Average hours watched: 11.28

Scores by platform:
                    Affordability  Content_Driven  UX_Impact  Retention  Recommendation
Platform                                                                               
Amazon Prime Video       3.800000        4.050000   3.400000   3.650000        3.650000
Disney+ Hotstar          3.500000        3.656250   3.500000   3.406250        3.968750
Hoichoi, Hotstar         3.000000        4.000000   3.000000   5.000000        5.000000
Netflix                  3.473684        3.842105   3.921053   3.736842        3.763158
No ott platform          4.000000        3.000000   3.000000   4.000000        4.000000
None, only Youtube       3.000000        3.000000   4.000000  

In [50]:
#System prompt

system_prompt = """
You are a Data Analyst AI Assistant.

You analyze OTT platform survey data.

Your job:
- Compare platforms
- Analyze retention, affordability, and user behavior
- Identify trends
- Give actionable insights

Be concise and data-driven.
"""

In [51]:
def ask_analyst(question, context):

    user_prompt = f"""
Dataset insights:
{context}

Question:
{question}
"""

    response = groq.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

In [52]:
#Test

print(ask_analyst(
    "Which OTT platform performs best overall and why?",
    data_context
))

To determine the overall best-performing OTT platform, I'll analyze the scores across Affordability, Content_Driven, UX_Impact, Retention, and Recommendation.

**Weighted Average Calculation:**
Since each aspect is equally important, I'll calculate a weighted average for each platform using the scores provided.

1. Amazon Prime Video: (3.8 + 4.05 + 3.4 + 3.65 + 3.65) / 5 = 3.73
2. Disney+ Hotstar: (3.5 + 3.656 + 3.5 + 3.406 + 3.969) / 5 = 3.606
3. Hoichoi, Hotstar: (3 + 4 + 3 + 5 + 5) / 5 = 4
4. Netflix: (3.473 + 3.842 + 3.921 + 3.737 + 3.763) / 5 = 3.737
5. Zee5: (3.667 + 4.333 + 3.667 + 2.667 + 4.333) / 5 = 3.733

**Ranking:**
Based on the weighted average, the ranking is:
1. Hoichoi, Hotstar (4)
2. Netflix (3.737)
3. Amazon Prime Video (3.73)
4. Zee5 (3.733)
5. Disney+ Hotstar (3.606)

**Best Overall Platform:**
Although Hoichoi, Hotstar has the highest weighted average, it's essential to consider the number of responses (only 1). This might not be representative of the overall user

In [53]:
#Smart Analyst

def smart_analyst(question):

    q = question.lower()

    if "highest retention" in q:
        result = df.groupby('Platform')['Retention'].mean().idxmax()
        context = f"Platform with highest retention: {result}"

    elif "most affordable" in q:
        result = df.groupby('Platform')['Affordability'].mean().idxmax()
        context = f"Most affordable platform: {result}"

    elif "best content" in q:
        result = df.groupby('Platform')['Content_Driven'].mean().idxmax()
        context = f"Best content platform: {result}"

    elif "most recommended" in q:
        result = df.groupby('Platform')['Recommendation'].mean().idxmax()
        context = f"Most recommended platform: {result}"

    else:
        context = data_context

    return ask_analyst(question, context)

In [54]:
#Test smart version

print(smart_analyst("Which platform has highest retention?"))
print(smart_analyst("Which platform is most affordable?"))
print(smart_analyst("Which platform has best content?"))

According to the dataset, there are two platforms tied for the highest retention: Hoichoi and Hotstar.
According to the dataset, there is no OTT platform that stands out as the most affordable. The data suggests that none of the platforms are significantly more affordable than the others.
According to the dataset insights, Zee5 has the best content among the OTT platforms.


In [55]:
#Gradio UI

import gradio as gr

# Chat function
def chat_fn(message, history):
    if history is None:
        history = []

    # Get response
    response = smart_analyst(message)

    # Append conversation
    history.append((message, response))

    return history, history

with gr.Blocks() as demo:
    gr.Markdown("# 📊 OTT Data Analyst Assistant")
    gr.Markdown("Ask anything about OTT user behavior, retention, pricing, etc.")

    chatbot = gr.Chatbot(height=450)

    msg = gr.Textbox(
        placeholder="Type your question and press Enter...",
        container=False
    )

    state = gr.State([])  # stores full chat history

    # Submit message
    msg.submit(
        chat_fn,
        inputs=[msg, state],
        outputs=[chatbot, state]
    )

# Enable queue (smooth UX like ChatGPT)
demo.queue()

# Launch
demo.launch(inbrowser=True)

C:\Users\KIIT\AppData\Local\Temp\ipykernel_26688\3430026567.py:22: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=450)


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.
